In [1]:
### Pretraining vs. Fine-tuning

Base (pretrained-only) vs. instruct (fine-tuned) Qwen2.5-0.5B run side by side on identical prompts.
! pip install torch==2.9.0 transformers==4.57.3 matplotlib==3.10.0 pandas==2.2.2 numpy==2.0.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 98.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.0
    Uninstalling huggingface_hub-1.4.0:
      Successfully uninstalled huggingface_hub-1.4.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_ID = "Qwen/Qwen2.5-0.5B"
INST_ID = "Qwen/Qwen2.5-0.5B-Instruct"

def load(model_id):
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype="auto",
        device_map="auto",
        trust_remote_code=True
    )
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    return tok, mdl

base_tok, base_m = load(BASE_ID)
inst_tok, inst_m = load(INST_ID)

@torch.inference_mode()
def gen_base(prompt, max_new_tokens=80):
    x = base_tok(prompt, return_tensors="pt").to(base_m.device)
    y = base_m.generate(
        **x,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=base_tok.eos_token_id,
    )
    return base_tok.decode(y[0], skip_special_tokens=True)

@torch.inference_mode()
def gen_instruct(user_prompt, max_new_tokens=80):
    # Proper chat formatting for instruct model
    chat = [{"role": "user", "content": user_prompt}]
    prompt = inst_tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    x = inst_tok(prompt, return_tensors="pt").to(inst_m.device)
    y = inst_m.generate(
        **x,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=inst_tok.eos_token_id,
    )
    out = inst_tok.decode(y[0], skip_special_tokens=True)
    # Trim the echoed prompt if present
    return out[len(inst_tok.decode(x["input_ids"][0], skip_special_tokens=True)) :].strip()

tests = [
    "How do I make a pizza?",
    "Write a haiku about summer.",
]

for p in tests:
    print("\n" + "="*90)
    print("PROMPT:", p)
    print("-"*90)
    print("BASE (pretrained):")
    print(gen_base(p))
    print("-"*90)
    print("INSTRUCT (fine-tuned):")
    print(gen_instruct(p))


### Second example: Llama-2 (base vs. chat) and ChatGPT

Same comparison, different model family and toolchain (Lamini API + local `llama.py` runner, not
self-contained like the Qwen example above — needs `POWERML__PRODUCTION__URL`/`KEY` env vars and a
`../utils/llama.py` module to actually execute).


In [12]:
import os
import lamini

lamini.api_url = os.getenv("POWERML__PRODUCTION__URL")
lamini.api_key = os.getenv("POWERML__PRODUCTION__KEY")

In [14]:
import sys
import os

sys.path.append('../utils')
#print(os.path.exists('../utils/llama.py'))  # Should print True
#print(os.listdir('../utils'))  # Shows files in the utils folder

from llama import BasicModelRunner

### Try Non-Finetuned models

In [15]:
non_finetuned = BasicModelRunner("meta-llama/Llama-2-7b-hf")

In [16]:
non_finetuned_output = non_finetuned("Tell me how to train my dog to sit")

In [17]:
print(non_finetuned_output)

Tell me how to train my dog to stay.
Tell me how to teach my dog to come.
Tell me how to get my dog to heel.
Tell me how to stop my dog from jumping.
Tell me how to make my dog stop barking.
Tell me how I can get my dog to stop chewing.
Tell me how to house train my dog.
Tell me how to potty train my dog.
Tell me how to train my dog to walk on a leash.
Tell me how to crate train my dog.
Tell Me How To Train My Dog To Sit, Stay, Come, Heel, Stop Jumping, Stop Barking, Stop Chewing, House Train, Potty Train, Walk On A Leash, Crate Train, And More!
Tell Me How To Train Your Dog To Sit, Stay, Heel, Come, Stop Jumping, Stop Chewing, House Trained, Potty Trained, Walk On A Leash, And More!
Tell me how to train your dog to sit.
Tell Me How To House Train My Dog.


In [18]:
print(non_finetuned("What do you think of Mars?"))

I think it's a great planet.
I think it's a good planet.
I think it'll be a great planet.
I think we should go there.
I think we should go back there.
I think we should stay there.
I think we should leave there.
I think we should colonize there.
I think we should terraform there.
I think we should mine there.
I think we should build there.
I think we should live there.
I think we should die there.
I think we should be there.
I think we should have been there.
I think we should never be there.
I think we'll be there.
I think we won't be there.
I think we will be there.
I think we can be there.
I think we could be there.
I think we would be there.
I think we might be there.
I think we may be there.
I think we shall be there.
I think we must be there.
I think we have to be there.
I think we need to be there.
I think I'll be there.
I know I'll be there.


In [19]:
print(non_finetuned("taylor swift's best friend"))

I'm not sure if I've mentioned this before, but I'm a huge Taylor Swift fan.
I've been a fan since her first album, and I've been a fan ever since. I've been a fan of her music, her style, her personality, and her music. I've been a fan for a long time.
I've been a fan of Taylor Swift for a long time. I've been a fan of her music for a long time. I'm a fan of her style, her personality, her music, and her music. I've been a fan for a long, long time.
I've always been a fan of Taylor Swift. I've always been a fan of her music.
I've always been a big fan of her style. I've always been a huge fan of her personality. I've always been a massive fan of her music. I'm a fan of Taylor Swift. I love her music. I love her style. I love her personality. I love her music. I'm a huge fan of Taylor Swift.
I've always been an avid fan of Taylor Swift. I'm


In [20]:
print(non_finetuned("""Agent: I'm here to help you with your Amazon deliver order.
Customer: I didn't get my item
Agent: I'm sorry to hear that. Which item was it?
Customer: the blanket
Agent:"""))

I'm sorry to hear that but I'm here to help.
Customer: I don't get the blanket
Customer: I don't have the blanket
Agent: I don't understand.
Agent: I'm sorry. I don't understand. customer: I don't have the blanket Agent: I'm so sorry to hear that.
Customer: I don t have the blanket
Agent: Oh no, I'm so sorry to hear about that.
customer: I


### Compare to finetuned models 

In [21]:
finetuned_model = BasicModelRunner("meta-llama/Llama-2-7b-chat-hf")

In [22]:
finetuned_output = finetuned_model("Tell me how to train my dog to sit")

In [23]:
print(finetuned_output)

on command. Training your dog to sit is a basic obedience command that can be achieved with patience, consistency, and positive reinforcement. Here's a step-by-step guide on how to train your dog to sit:

1. Choose a quiet and distraction-free area: Find a quiet room or area where your dog can focus on you without getting distracted.
2. Have treats ready: Choose your dog's favorite treats and have them readily available to reward good behavior.
3. Stand in front of your dog: Stand in front of your dog and hold a treat close to their nose.
4. Move the treat up and back: Slowly move the treat upwards and backwards, towards your dog's tail, while saying "sit" in a calm and clear voice.
5. Dog will follow the treat: As you move the treat, your dog should naturally sit down to follow it. The moment they touch their bottom to the ground, give them the treat and praise them.


In [24]:
print(finetuned_model("[INST]Tell me how to train my dog to sit[/INST]"))

Training your dog to sit is a basic obedience command that can be achieved with patience, consistency, and positive reinforcement. Here's a step-by-step guide on how to train your dog to sit:

1. Choose a quiet and distraction-free area: Find a quiet room or area where your dog can focus on you without getting distracted.
2. Have treats ready: Choose your dog's favorite treats and have them readily available to reward good behavior.
3. Stand in front of your dog: Stand in front of your dog and hold a treat close to their nose.
4. Move the treat up and back: Slowly move the treat upwards and backwards, towards your dog's tail, while saying "sit" in a calm and clear voice.
5. Dog will follow the treat: As you move the treat, your dog should naturally sit down to follow it. The moment they touch their bottom to the ground, give them the treat and praise them.
6. Repeat the process: Repeat steps 3-5 several times, so your dog learns to associate the command "sit" with the action of sitting

In [25]:
print(non_finetuned("[INST]Tell me how to train my dog to sit[/INST]"))

[INST]Tell me how to train my dog to sit[/INST]


In [26]:
print(finetuned_model("What do you think of Mars?"))

Mars is a fascinating planet that has captured the imagination of humans for centuries. It is the fourth planet from the Sun in our solar system and is kn own for its reddish appearance. Mars is a rocky planet with a thin atmosphere, and its surface is characterized by volcanoes, canyons, and impact crater.
One of the most intriguing aspects of Mars is its potential for supporting life. While there is currently no evidence of life on Mars, the planet's atmosphere and geology suggest that it may have been habitable in the past. NASA's Curiosity rover has been exploring Mars since 2012, and has discovered evidence of water on the planet, which is a key ingredient for life.
Mars is also a popular target for space missions and future human settlements. Several space agencies and private companies are planning missions to Mars in the coming years, with the goal of establishing a human presence on the pl anet. The challenges of establishing a human settlement on Mars are significa nt, includ

In [27]:
print(finetuned_model("taylor swift's best friend"))

Taylor Swift's best friend is a person who has been by her side through thick and thin. Here are some possible candidates:
1. Abigail Anderson - Abigail is Taylor's childhood friend and has been a constant presence in her life. The two have been inseparable since they met in kindergarten and have shared countless memories together.
2. Selena Gomez - Selena and Taylor have been friends for over a decade and have been through a lot together. They have collaborated on music projects, gone on vacation together, and have been there for each other through personal struggles.
3. Liz Rose - Liz is a songwriter and producer who has worked with Taylor on many of her hit songs. The two have a close creative partnership and have been friends for many years.
4. Joe Jonas - Joe and Taylor were friends for several years before they started dating in 2008. They have been on-again, off-again since then and have been there for each other during some of the most significant moments in each other's lives.

In [28]:
print(finetuned_model("""Agent: I'm here to help you with your Amazon deliver order.
Customer: I didn't get my item
Agent: I'm sorry to hear that. Which item was it?
Customer: the blanket
Agent:"""))

I see. Can you please provide me with your order number so I can look into this for you?
Customer: I don't have the order number.
Agent: Okay, no worries. Can you please tell me the name of the item you didn't receive?
Customer: Blanket
Agent: And what was the shipping address for this item?
Customer: 123 Main St
Agent: Okay, thank you. I'm going to check on the status of your order. Can you please hold for just a moment?
Customer: (hangs up)
This is a common scenario in customer service, where the agent is trying to gather information from the customer to help resolve their issue. The agent is using active listening skills to gather information from the customer, such as the order number, the name of the item, and the shipping address. The agent is also using empathy to acknowledge the customer's concern and let them know that they are there to help.


### Compare to ChatGPT

> Note: This section of the notebook has been updated.

When run with the query:

```Tell me how to train my dog to sit```

ChatGPT responds with:

> Training your dog to sit is a basic and essential command that can be taught using positive reinforcement. Here's a simple step-by-step guide:
> 1. **Prepare Treats:**
   Gather small, soft treats that your dog enjoys. Make sure they are easy to chew and won't take too long to eat.
> 2. **Find a Quiet Space:**
   Choose a quiet area with minimal distractions for the training session. This will help your dog focus better.
> 3. **Get Your Dog's Attention:**
   Call your dog's name to get their attention. Make sure they are looking at you.
> 4. **Use a Treat to Lure:**
   Hold a treat close to your dog's nose, and slowly move your hand upward and slightly backward over their head. As you do this, your dog's natural response will be to follow the treat with their nose, causing them to sit.
> 5. **Say the Command:**
   As your dog starts to sit, say the command "Sit" in a clear and firm voice. Use the word consistently every time you want your dog to sit.
> 6. **Reward and Praise:**
   As soon as your dog sits, immediately reward them with the treat and offer verbal praise. This positive reinforcement will help them associate sitting with positive outcomes.
> 7. **Repeat and Practice:**
   Repeat the process several times in a row during each training session. Keep the sessions short (around 5-10 minutes) to prevent your dog from losing interest.
> 8. **Add Duration:**
   Once your dog consistently sits on command, you can gradually increase the duration by waiting a couple of seconds before giving the treat. This helps reinforce the sit command.
> 9. **Generalize the Command:**
   Practice the "sit" command in different locations and with various distractions to help your dog generalize the behavior.
> 10. **Be Patient and Consistent:**
    Patience and consistency are key in dog training. Always use positive reinforcement, and avoid punishment. If your dog doesn't succeed initially, go back a step and try again.
> 
> Remember that each dog is unique, and some may learn more quickly than others. Adjust your training approach based on your dog's individual needs and progress.